In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.eval.llama_evaluator import LLAMA2_PROMPT

texts = [
    ("hi!", "Hello! How can I assist you today?"),
    ("What is the capital of France?", "The capital of France is Paris."),
    ("Can you tell me a joke?", "Sure! Why don't scientists trust atoms? Because they make up everything!"),
    ("What is the meaning of life?", "The meaning of life is subjective and can vary from person to person."),
    ("Tell me a story.", "Once upon a time in a faraway land, there lived a brave knight who fought dragons and saved villages."),
    (
        "How do I cook pasta?",
        "To cook pasta, boil water, add salt, then add the pasta and cook until al dente. Drain and serve with your favorite sauce.",
    ),
]

formated = [LLAMA2_PROMPT.format(behavior=prm, generation=gen) for (prm, gen) in texts]

conversations = [[{"role": "user", "content": LLAMA2_PROMPT.format(behavior=prm, generation=gen)}] for (prm, gen) in texts]

In [ ]:
from gserve import LLMConfig, ServeConfig
from gserve.vllm_service import VLLMService
from vllm import SamplingParams

model_name = "meta-llama/Llama-3.1-8B-Instruct"

llm_config = LLMConfig(
    model_name=model_name,
    dtype="bfloat16",
)

serve_config = ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=2 * 60, verbose=True)

model = VLLMService(llm_config, serve_config)
model.start()

In [ ]:
sampling_params = SamplingParams(
    temperature=0,
    top_p=1.0,
    top_k=-1,
    repetition_penalty=1.0,
    max_tokens=40,
)

output = model.generate(formated, sampling_params=sampling_params)

for out in output:
    print(out)

In [ ]:
# now lets use transformers to do the same
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_transformers = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

In [ ]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_transformers,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16, "skip_special_tokens": True, "add_generation_prompt": True, "padding_side": "left"},
    max_new_tokens=40,
    device="cuda:0",
    pad_token_id=128001,
    eos_token_id=128001,
)

batch_output = pipeline(
    formated,
    return_full_text=False,
    do_sample=False,
    top_p=1.0,
    top_k=-1,
    temperature=0,
    use_model_defaults=True,
    pad_token_id=128001,
    eos_token_id=128001,
    repetition_penalty=1,
)

for out in batch_output:
    print(out)

In [ ]:
raise "aaaaa"

----------------------

In [1]:
import torch
import numpy as np

# 1. Common settings
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42
TEMPERATURE = 0.0
TOP_K = 0
TOP_P = 1.0
REPETITION_PENALTY = 1.1
MAX_NEW_TOKENS = 128
PROMPT = "How do you do, where are you, blah afaf comlet, "

# fix seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

In [2]:
# 2. vLLM setup
from vllm import LLM, SamplingParams

vllm_llm = LLM(
    model=MODEL_NAME,
    seed=SEED,
    dtype="bfloat16",
)

INFO 06-11 19:42:11 [__init__.py:243] Automatically detected platform cuda.
INFO 06-11 19:42:15 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-11 19:42:15 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-11 19:42:15 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-11 19:42:27 [config.py:793] This model supports multiple tasks: {'reward', 'embed', 'score', 'classify', 'generate'}. Defaulting to 'generate'.
INFO 06-11 19:42:27 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-11 19:42:29 [core.py:438] Waiting for init message from front-end.
INFO 06-11 19:42:29 [core.py:65] Initializing a V1 LLM engine (v0.9.0.1) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 06-11 19:42:36 [default_loader.py:280] Loading weights took 3.83 seconds
INFO 06-11 19:42:37 [gpu_model_runner.py:1549] Model loading took 14.9889 GiB and 5.117977 seconds
INFO 06-11 19:42:45 [backends.py:459] Using cache directory: /home/fre.gilad/.cache/vllm/torch_compile_cache/97dfb5a5e5/rank_0_0 for vLLM's torch.compile
INFO 06-11 19:42:45 [backends.py:469] Dynamo bytecode transform time: 8.89 s
INFO 06-11 19:42:55 [backends.py:132] Directly load the compiled graph(s) for shape None from the cache, took 9.569 s
INFO 06-11 19:42:56 [monitor.py:33] torch.compile takes 8.89 s in total
INFO 06-11 19:43:01 [kv_cache_utils.py:637] GPU KV cache size: 213,200 tokens
INFO 06-11 19:43:01 [kv_cache_utils.py:640] Maximum concurrency for 131,072 tokens per request: 1.63x
INFO 06-11 19:43:30 [gpu_model_runner.py:1933] Graph capturing finished in 29 secs, took 1.59 GiB
INFO 06-11 19:43:30 [core.py:167] init engine (profile, create kv cache, warmup model) took 53.46 seconds


In [13]:
vllm_sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    max_tokens=MAX_NEW_TOKENS,
)

# generate with vLLM
vllm_resp = vllm_llm.generate(
    PROMPT,
    sampling_params=vllm_sampling_params,
)

vllm_output = vllm_resp[0].outputs[0].text
print("vLLM output:\n", vllm_output)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

vLLM output:
 1st time here. I'm a bit of a noob so bear with me.
Welcome to the forums! Don't worry about being a "noob", we're all here to learn and have fun. What brings you to this place? Are you looking for help with something specific or just wanting to chat with fellow gamers?

Also, feel free to ask if you need any help navigating the forums or finding resources. We've got a lot of great content and community members who are happy to assist.

So, what's your story? What kind of games are you into, and how did you find us?

(And don't


In [4]:
# 3. Transformers pipeline setup
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline, GenerationConfig

# load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    add_generation_prompt=True,
    padding_side="left",
    use_fast=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda:1",
)

# ensure pad_token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
# generation config matching vLLM SamplingParams
gen_config = GenerationConfig(
    do_sample=False,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    max_new_tokens=MAX_NEW_TOKENS,
)

# create a torch Generator with the same seed
generator = torch.Generator(device=model.device).manual_seed(SEED)

pipeline = TextGenerationPipeline(
    model=model.eval(),
    tokenizer=tokenizer,
    generation_config=gen_config,
)

# generate with transformers
hf_outputs = pipeline(
    PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    num_return_sequences=1,
    return_full_text=True,
    use_model_defaults=True,
)

hf_output = hf_outputs[0]["generated_text"]
print("\nTransformers output:\n", hf_output)

/home/fre.gilad/source/llm-iml/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/fre.gilad/source/llm-iml/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
Device set to use cuda:1
/


Transformers output:
 How do you do, where are you, blah afaf comlet, 1st time here. I'm a bit of a noob so bear with me.
Welcome to the forums! Don't worry about being a "noob", we're all here to help and learn from each other. What brings you to this place? Are you looking for information on a specific topic or just wanting to chat with others who share similar interests?

Also, feel free to ask if you need any help navigating the site or figuring out how things work around here.

(And don't worry, I won't hold it against you that you said "blah afaf comlet" - it's not every day someone uses that
